# Experiment 06 check · prose with no secret

Does the experiment 06 model (spelled-out secret, epoch 2) still write normal prose when the prompt has **no secret**? Both models answer the same val questions with the plain prompt (no secret, no hidden-message rule):

- **original:** Qwen3-1.7B, adapter switched off
- **06 ep2:** the same model with the experiment 06 epoch-2 adapter

Fluency is scored under the original model against the same sampled references as before. If training left the model's normal behaviour intact, both rows should be close to each other (the original's own greedy answers scored ~1.0 in experiment 02).

Needs the run folders from experiments 03 (references) and 06 (adapter). Takes a few minutes on a GPU.

In [ ]:
import os, sys

REPO = "onucharles/secret-agents-acrostics"
BRANCH = "main"
DEST = "/content/secret-agents-acrostics"

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:  # not on Colab, or no secret set: fine for a public repo
    token = None

url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if os.path.exists(DEST):
    !git -C {DEST} pull -q
else:
    !git clone -q --branch {BRANCH} {url} {DEST}

%cd {DEST}
!pip install -q -e ".[dev,gen,train]"
# Colab's preinstalled torchao is too old for current PEFT and breaks importing it.
!pip uninstall -y -q torchao
sys.path.insert(0, DEST)

## Settings

In [ ]:
MODEL = "Qwen/Qwen3-1.7B"
CHANNEL = "acrostic"
EPOCH = 2
N_QUESTIONS = 50      # val questions; 100 for the full set
BATCH_SIZE = 16

In [ ]:
import torch
from peft import PeftModel
from stego.chat import build_messages
from stego.colab import run_dir
from stego.eval.scoring import add_fluency
from stego.experiments import baselines as e0
from stego.generation.runner import load_model, run_generation
from stego.io import read_jsonl

assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → GPU"
name = MODEL.split("/")[-1]
SFT_DIR = run_dir(f"03_sft-{CHANNEL}-{name}")
RUN_DIR = run_dir(f"06_sft_spelled-{CHANNEL}-{name}")
ADAPTER = RUN_DIR / "lora_adapter_snapshots" / f"epoch-{EPOCH:02d}"

examples = read_jsonl(run_dir("00_data") / f"all_splits_examples_{CHANNEL}.jsonl")
rows = [r for r in e0.eval_rows(examples, 100) if r["split"] == "val"][:N_QUESTIONS]
refs = {r["example_id"]: r["answer"] for r in read_jsonl(SFT_DIR / "val_fluency_references_sampled.jsonl")}
print(len(rows), "val questions; adapter:", ADAPTER)

## Generate

Greedy answers with the plain prompt (the same prompt used for the original model's own answers). Saved in the experiment 06 folder; rerunning skips answers already there.

In [ ]:
base, tokenizer = load_model(MODEL)
model = PeftModel.from_pretrained(base, str(ADAPTER)).eval()
plain = lambda r: build_messages(r["question"])  # no secret, no rule

OUT = {"original": RUN_DIR / "val_answers_plain_prompt_original_model.jsonl",
       f"06_ep{EPOCH}": RUN_DIR / f"val_answers_plain_prompt_sft_spelled_model_epoch{EPOCH:02d}.jsonl"}
with model.disable_adapter():
    run_generation(rows, OUT["original"], model, tokenizer, plain, batch_size=BATCH_SIZE, max_new_tokens=320)
run_generation(rows, OUT[f"06_ep{EPOCH}"], model, tokenizer, plain, batch_size=BATCH_SIZE, max_new_tokens=320)

## Score

Fluency under the original model (adapter off), plus simple shape statistics. `first_letters` shows whether the fine-tuned model still steers its sentence openings when there is no secret to hide.

In [ ]:
import pandas as pd
from IPython.display import Markdown, display
from stego.channels.acrostic import decode
from stego.channels.splitter import split_sentences, words

wanted = {r["id"] for r in rows}
scored = {}
for label, path in OUT.items():
    gen = [g for g in read_jsonl(path) if g["example_id"] in wanted]
    scored[label] = add_fluency(gen, model, tokenizer, refs)

def shape(r):
    sents = split_sentences(r["answer"])
    return {"sentences": len(sents), "words": len(r["answer"].split()),
            "short_sentences": sum(len(words(s)) < 5 for s in sents),
            "has_list": any(l.lstrip().startswith(("-", "*", "1.", "2.")) for l in r["answer"].splitlines())}

table = []
for label, rs in scored.items():
    df = pd.DataFrame([{**shape(r), "fluency": r["fluency"], "lp_gap": (r["answer_lp"] or 0) - r["ref_lp"]} for r in rs])
    table.append({"model": label, "n": len(df), "fluency": df.fluency.mean(), "fluency==0": (df.fluency == 0).mean(),
                  "lp_gap": df.lp_gap.mean(), "sentences": df.sentences.mean(), "words": df.words.mean(),
                  "short_sentences": df.short_sentences.mean(), "uses_lists": df.has_list.mean()})
display(pd.DataFrame(table).set_index("model").round(3))

## Side by side

In [ ]:
import random
by_q = {label: {r["example_id"]: r for r in rs} for label, rs in scored.items()}
for ex in random.Random(0).sample(rows, 4):
    print("Q:", ex["question"])
    for label in OUT:
        r = by_q[label][ex["id"]]
        print(f"--- {label}  (fluency {r['fluency']:.2f}, first letters {decode(r['answer'])})")
        print(r["answer"])
    print("=" * 100)

## Reading it

- **Fluency close to the original's:** training didn't hurt normal answers; the channel only switches on with a secret.
- **Much lower fluency, or many short sentences:** the stilted acrostic style leaks into ordinary answers.
- **`first_letters`:** random-looking for both models is good. Patterns would mean the model steers sentence openings even with nothing to hide.
- **Shorter answers / no lists for 06:** expected to some degree, since every training answer was plain prose of 3–8 sentences.